# Lista 3 - KNN e Árvore de Decisão

## Questão 1

Considere o conjunto de dados disponível em **kc2.csv**, organizado em 22 colunas, sendo as 21 primerias colunas os atributos e a última coluna a saída. Os 21 atributos são referentes à caracterização de códigos-fontes para processamento de dados na NASA. A saída é a indicação de ausência (0) ou existência (1) de defeitos (os dados foram balanceados via sub amostragem).

### a) Considerando uma validação cruzada em 10 *folds*, avalie modelos de classificação binária nos dados em questão. Para tanto, use as abordagens abaixo:
* **KNN** (escolha $k = 1$ e $k = 5$, distância Euclidiana e Mahalonobis, totalizando 4 combinações);
* **Árvore de decisão** (você pode usar uma implementação já existente, como a do scikit-learn, com índices de impureza de gini e entropia).


### **Solução**

In [1]:
import sys
sys.path.append("../")

#### k-NN

In [2]:
# 1. Importação das bibliotecas necessárias
from utils.utils import *
from models.knn import kNN
from preprocessors.normalizador import Normalizador
from metrics.matrizdeconfusao import MatrizDeConfusaoBinaria

In [3]:
# 2. Extração dos dados
X = kc2[:, :-1]
y = kc2[:, [-1]]

In [4]:
# 3. Divisão dos dados nos conjuntos de treinamento e teste
X_train, X_test, y_train, y_test = treine_teste_divida(X, y)

In [6]:
# 4. Normalização dos dados de treinamento via z-score
normalizadorKNN = Normalizador(X_train, "StandardScaler")
X_train_normalizado = normalizadorKNN.normaliza(X_train)

In [5]:
# 5. Inicialização dos modelos k-NN
modelo_knn_1_euclidiana = kNN(1, "euclidiana")
modelo_knn_1_mahalanobis = kNN(1, "mahalanobis")
modelo_knn_5_euclidiana = kNN(5, "euclidiana")
modelo_knn_5_mahalanobis = kNN(5, "mahalanobis")

modelos = {"modelo_knn_1_euclidiana":modelo_knn_1_euclidiana, 
           "modelo_knn_1_mahalanobis":modelo_knn_1_mahalanobis, 
           "modelo_knn_5_euclidiana":modelo_knn_5_euclidiana, 
           "modelo_knn_5_mahalanobis":modelo_knn_5_mahalanobis}

modelos_keys = list(modelos.keys())
modelos_objeto = list(modelos.values())

In [6]:
# 6. Avaliação via k-fold cross validation (k = 10) e cálculo das métricas de avaliação
Xy = np.hstack([X_train_normalizado, y_train])
reports = {}
for i in modelos_keys:
    reports[i] = {}

for i, modelo in modelos.items():
    reports[i] = {}
    folds = kfold(Xy, 10)
    acuracias = []
    revocacoes = []
    precisoes = []
    f1_scores = []
    for f in range(10):
        valid_fold = folds.pop()
        train_fold = np.vstack(folds)
        modelo.ajuste(train_fold[:,:-1], train_fold[:, [-1]])
        y_pred = modelo.prever(valid_fold[:,:-1]).reshape(-1,1)
        y_real = valid_fold[:, [-1]]
        folds.insert(0, valid_fold)
        matriz = MatrizDeConfusaoBinaria(y_real, y_pred)
        acuracias.append(matriz.acuracia())
        revocacoes.append(matriz.revocacao())
        precisoes.append(matriz.precisao())
        f1_scores.append(matriz.f1_score())
    reports[i]["acuracia_media"] = np.mean(np.array(acuracias))
    reports[i]["acuracia_desvio"] = np.std(np.array(acuracias))
    reports[i]["revocacao_media"] = np.mean(np.array(revocacoes))
    reports[i]["revocacao_desvio"] = np.std(np.array(revocacoes))
    reports[i]["precisao_media"] = np.mean(np.array(precisoes))
    reports[i]["precisao_desvio"] = np.std(np.array(precisoes))
    reports[i]["f1-score_media"] = np.mean(np.array(f1_scores))
    reports[i]["f1-score_desvio"] = np.std(np.array(f1_scores))

#### Árvore de Decisão

In [7]:
# 1. Importação das bibliotecas necessárias
from utils.utils import *
from models.arvorededecisao import ArvoreDeDecisao
from metrics.matrizdeconfusao import MatrizDeConfusaoBinaria

In [8]:
# 2. Extração dos dados
X = kc2[:, :-1]
y = kc2[:, [-1]]

In [9]:
X.shape

(214, 21)

In [10]:
# 3. Divisão dos dados nos conjuntos de treinamento e teste
X_train, X_test, y_train, y_test = treine_teste_divida(X, y)

In [7]:
# 4. Inicialização das árvores
# OBS: Para evitar overfitting, foi posto um valor de impureza mínima para as folhas.
modelo_arvore_de_decisao_gini = ArvoreDeDecisao("gini", 0.2)
modelo_arvore_de_decisao_entropia = ArvoreDeDecisao("entropia", 0.2)

modelos = {"modelo_arvore_de_decisao_gini":modelo_arvore_de_decisao_gini, 
           "modelo_arvore_de_decisao_entropia":modelo_arvore_de_decisao_entropia}

modelos_keys = list(modelos.keys())
modelos_objeto = list(modelos.values())

In [8]:
# 5. Avaliação via k-fold cross validation (k = 10) e cálculo das métricas de avaliação
Xy = np.hstack([X_train, y_train])
reports = {}
for i in modelos_keys:
    reports[i] = {}

for i, modelo in modelos.items():
    reports[i] = {}
    folds = kfold(Xy, 10)
    acuracias = []
    revocacoes = []
    precisoes = []
    f1_scores = []
    for f in range(10):
        valid_fold = folds.pop()
        train_fold = np.vstack(folds)
        modelo.ajuste(train_fold[:,:-1], train_fold[:, [-1]])
        y_pred = modelo.prever(valid_fold[:,:-1]).reshape(-1,1)
        y_real = valid_fold[:, [-1]]
        folds.insert(0, valid_fold)
        matriz = MatrizDeConfusaoBinaria(y_real, y_pred)
        acuracias.append(matriz.acuracia())
        revocacoes.append(matriz.revocacao())
        precisoes.append(matriz.precisao())
        f1_scores.append(matriz.f1_score())
    reports[i]["acuracia_media"] = np.mean(np.array(acuracias))
    reports[i]["acuracia_desvio"] = np.std(np.array(acuracias))
    reports[i]["revocacao_media"] = np.mean(np.array(revocacoes))
    reports[i]["revocacao_desvio"] = np.std(np.array(revocacoes))
    reports[i]["precisao_media"] = np.mean(np.array(precisoes))
    reports[i]["precisao_desvio"] = np.std(np.array(precisoes))
    reports[i]["f1-score_media"] = np.mean(np.array(f1_scores))
    reports[i]["f1-score_desvio"] = np.std(np.array(f1_scores))

### b) Para cada modelo criado, reporte valor médio e desvio padrão das métricas de **acurácia**, **revocação**,  **precisão** e **F1-score**.

### **Solução**

#### k-NN

In [13]:
# 7. Reporte dos valores médios e desvios padrões das métricas de cada modelo k-NN criado.
for modelo, metricas in reports.items():
    print(f"================================= Resultados do {modelo} =================================")
    for nome, valor in metricas.items():
        print(f"{nome}: {valor}")
    print()

================================= Resultados do modelo_knn_1_euclidiana =================================
acuracia_media: 0.75
acuracia_desvio: 0.07954345035153529
revocacao_media: 0.7602344877344878
revocacao_desvio: 0.09799365062461612
precisao_media: 0.7570326895326895
precisao_desvio: 0.07893367951159251
f1-score_media: 0.7522458047777743
f1-score_desvio: 0.056221794598545226

================================= Resultados do modelo_knn_1_mahalanobis =================================
acuracia_media: 0.7555555555555555
acuracia_desvio: 0.0753592220347252
revocacao_media: 0.7962265512265512
revocacao_desvio: 0.08270157164293972
precisao_media: 0.7435497835497836
precisao_desvio: 0.07638267631912717
f1-score_media: 0.7657121172601049
f1-score_desvio: 0.0627752680325017

================================= Resultados do modelo_knn_5_euclidiana =================================
acuracia_media: 0.7944444444444445
acuracia_desvio: 0.11124991330278215
revocacao_media: 0.7843939393939394
revoca

#### Árvore de Decisão

In [9]:
# 7. Reporte dos valores médios e desvios padrões das métricas de cada modelo de árvore de decisão (gini e entropia) criado.
for modelo, metricas in reports.items():
    print(f"================================= Resultados do {modelo} =================================")
    for nome, valor in metricas.items():
        print(f"{nome}: {valor}")
    print()

================================= Resultados do modelo_arvore_de_decisao_gini =================================
acuracia_media: 0.75
acuracia_desvio: 0.08695819912499181
revocacao_media: 0.7682936507936506
revocacao_desvio: 0.13704173617098875
precisao_media: 0.751527639027639
precisao_desvio: 0.12112177772958735
f1-score_media: 0.7464637136319058
f1-score_desvio: 0.08460512863930958

================================= Resultados do modelo_arvore_de_decisao_entropia =================================
acuracia_media: 0.7388888888888889
acuracia_desvio: 0.08975274678557507
revocacao_media: 0.7497619047619047
revocacao_desvio: 0.12144558444984038
precisao_media: 0.7461507936507936
precisao_desvio: 0.1563647210495836
f1-score_media: 0.7321365914786967
f1-score_desvio: 0.09732358119102034



In [10]:
# 8. Avaliação via hold-out + visualização da árvore de decisão
# OBS: Apenas um passo extra para poder visualizar os nós de cada árvore criada

# Avaliação
modelos["modelo_arvore_de_decisao_gini"].ajuste(X_train, y_train)
y_pred = modelos["modelo_arvore_de_decisao_gini"].prever(X_test)
matriz_gini = MatrizDeConfusaoBinaria(y_test, y_pred)
print("===== Matriz de confusão (gini) =====")
print(matriz_gini)
print()

modelos["modelo_arvore_de_decisao_entropia"].ajuste(X_train, y_train)
y_pred = modelos["modelo_arvore_de_decisao_entropia"].prever(X_test)
matriz_entropia = MatrizDeConfusaoBinaria(y_test, y_pred)
print("===== Matriz de confusão (entropia) =====")
print(matriz_entropia)

===== Matriz de confusão (gini) =====
[[15  5]
 [ 1 22]]

===== Matriz de confusão (entropia) =====
[[15  5]
 [ 3 20]]


In [11]:
print("===== Árvore de Decisão (gini) =====")
print(f"Acurácia: {matriz_gini.acuracia()}")
print(f"Revocação: {matriz_gini.revocacao()}")
print(f"Precisão: {matriz_gini.precisao()}")
print(f"F1-score: {matriz_gini.f1_score()}")
print()

print("===== Árvore de Decisão (entropia) =====")
print(f"Acurácia: {matriz_entropia.acuracia()}")
print(f"Revocação: {matriz_entropia.revocacao()}")
print(f"Precisão: {matriz_entropia.precisao()}")
print(f"F1-score: {matriz_entropia.f1_score()}")

===== Árvore de Decisão (gini) =====
Acurácia: 0.8604651162790697
Revocação: 0.9565217391304348
Precisão: 0.8148148148148148
F1-score: 0.8800000000000001

===== Árvore de Decisão (entropia) =====
Acurácia: 0.813953488372093
Revocação: 0.8695652173913043
Precisão: 0.8
F1-score: 0.8333333333333333


In [12]:
# Visualização simplificada
print("===== Visualização simples da Árvore de Decisão (gini) =====")
modelos["modelo_arvore_de_decisao_gini"].visualizar_simplificado()

===== Visualização simples da Árvore de Decisão (gini) =====
d17 <= 16.0
	d10 <= 0.11
		d4 <= 42.0
			d17 <= 9.0
				d7 <= 6.0
					d16 <= 5.0
						d17 <= 6.0
							d8 <= 9.0
								d0 <= 7.0
									d0 <= 4.0
										d7 <= 1.5
											d0 <= 2.0
												Classe = 0
												d0 <= 3.0
													Classe = 0
													Classe = 0
											d0 <= 1.0
												Classe = 0
												Classe = 1
										Classe = 0
									Classe = 0
								Classe = 0
							Classe = 1
						Classe = 0
					d0 <= 18.0
						d0 <= 10.0
							Classe = 0
							Classe = 1
						Classe = 0
				d0 <= 14.0
					Classe = 1
					Classe = 0
			Classe = 0
		d5 <= 418.34
			Classe = 1
			Classe = 0
	d17 <= 25.0
		d8 <= 42.17
			d19 <= 64.0
				d8 <= 22.58
					Classe = 0
					Classe = 1
				d0 <= 68.0
					Classe = 0
					Classe = 1
			d6 <= 0.05
				Classe = 0
				d4 <= 115.0
					Classe = 0
					Classe = 1
		Classe = 1


In [13]:
# Visualização simplificada
print("===== Visualização simples da Árvore de Decisão (entropia) =====")
modelos["modelo_arvore_de_decisao_entropia"].visualizar_simplificado()

===== Visualização simples da Árvore de Decisão (entropia) =====
d17 <= 16.0
	d10 <= 0.11
		d4 <= 42.0
			d17 <= 9.0
				d7 <= 6.0
					d16 <= 5.0
						d17 <= 6.0
							d8 <= 9.0
								d0 <= 7.0
									d0 <= 4.0
										d0 <= 2.0
											Classe = 0
											d7 <= 1.5
												d0 <= 3.0
													Classe = 0
													Classe = 0
												Classe = 1
										Classe = 0
									Classe = 0
								Classe = 0
							Classe = 1
						Classe = 0
					d0 <= 18.0
						d0 <= 10.0
							Classe = 0
							Classe = 1
						Classe = 0
				d0 <= 14.0
					Classe = 1
					Classe = 0
			Classe = 0
		d5 <= 418.34
			Classe = 1
			Classe = 0
	d19 <= 117.0
		d4 <= 294.0
			d2 <= 4.0
				d6 <= 0.05
					d5 <= 860.0
						Classe = 0
						d0 <= 81.0
							Classe = 1
							Classe = 0
					d7 <= 13.89
						d0 <= 52.0
							d4 <= 103.0
								Classe = 1
								d0 <= 47.0
									Classe = 0
									Classe = 1
							Classe = 0
						Classe = 1
				Classe = 1
			Classe = 0
		C

In [14]:
# Visualização completa
print("===== Visualização completa da Árvore de Decisão (gini) =====")
modelos["modelo_arvore_de_decisao_gini"].visualizar()

===== Visualização completa da Árvore de Decisão (gini) =====
              --------------------
              |                  |
              |   d17 <= 16.0      |
              |   gini = 0.500   |
              |   N = 171         |
              |   [87, 84]         |
              |   Classe = 0     |
              |                  |
              --------------------
            
	              --------------------
              	|                  |
              	|   d10 <= 0.11      |
              	|   gini = 0.330   |
              	|   N = 91         |
              	|   [72, 19]         |
              	|   Classe = 0     |
              	|                  |
              	--------------------
            
		              --------------------
              		|                  |
              		|   d4 <= 42.0      |
              		|   gini = 0.291   |
              		|   N = 85         |
              		|   [70, 15]         |
              		|   Classe = 0     |
  

In [15]:
# Visualização completa
print("===== Visualização completa da Árvore de Decisão (entropia) =====")
modelos["modelo_arvore_de_decisao_entropia"].visualizar()

===== Visualização completa da Árvore de Decisão (entropia) =====
              --------------------
              |                  |
              |   d17 <= 16.0      |
              |   entropia = 1.000   |
              |   N = 171         |
              |   [87, 84]         |
              |   Classe = 0     |
              |                  |
              --------------------
            
	              --------------------
              	|                  |
              	|   d10 <= 0.11      |
              	|   entropia = 0.739   |
              	|   N = 91         |
              	|   [72, 19]         |
              	|   Classe = 0     |
              	|                  |
              	--------------------
            
		              --------------------
              		|                  |
              		|   d4 <= 42.0      |
              		|   entropia = 0.672   |
              		|   N = 85         |
              		|   [70, 15]         |
              		|   Cla